# From linear classification to neural networks

**Lab 5 · Building and fitting a neural network**

In Labs 3 and 4, you implemented predictors and classifiers, fitted their parameters,
and examined their empirical losses. Here you will do the same for a neural network.
We supply the familiar affine classifier on a new dataset. **You will implement the
network, use automatic differentiation to obtain its gradients, adapt gradient
descent, and plot and interpret the results.** Start with two hidden neurons, as
in the lecture's XOR example, then widen the network to 100 hidden neurons.
Finally, build a network with **two hidden layers of 5 and 7 neurons**, as in the
lecture's deeper architecture. Compare all three fits on the same observations.

Use **Google Colab** and work from top to bottom; no GPU is needed. At each exercise,
write your code in the marked cell before opening **Show answer**. The implementation
answers give complete solutions to copy into the corresponding exercise cell if
needed. **Plotting answers give only the missing lines:** insert them between
`# YOUR CODE START` and `# YOUR CODE END`, keeping all supplied code. Opening an
answer does not execute it; rerun the exercise cell after adding your code.

New PyTorch operations are explained before you need them. The data generation,
affine comparison and technical machinery for drawing boundaries are supplied.

We use the functions, loops, lists, tensor indexing and matrix products from
Labs 1--4. Short reminders below introduce the extra operations needed here.
Cells marked **Supplied** provide setup or checks; you do not need to recreate
them. Cells marked **YOUR CODE** are where you construct your own calculations.
Each exercise uses definitions made earlier, so complete it before running the
next checks. The visible instructions contain the tools needed to attempt every
exercise; opening a worked answer is optional.

## 1. Generate and visualise the data

We generate **100 observations**, arranged in four Gaussian clouds. Each cloud has 25 observations, with independent coordinate noise of standard deviation $0.45$.

| Cloud centre | Class | Observations |
|---|---:|---:|
| $(-1,-1)$ | 0 | 25 |
| $(1,1)$ | 0 | 25 |
| $(-1,1)$ | 1 | 25 |
| $(1,-1)$ | 1 | 25 |

Diagonally opposite clouds share a label. This is a noisy version of the XOR arrangement from the lectures: the inputs are points in the Euclidean plane, and each label records the class assigned to the cloud that generated the observation. Blue circles show class 0 and orange triangles show class 1, as in Lab 4.

The tensor `X` contains the two input coordinates and has shape `(N, 2)`. The labels in `y` have shape `(N, 1)`. Run the supplied cell to create and visualise the data.

In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Four Gaussian clouds in an XOR arrangement: 25 observations per cloud.
# spread is the standard deviation of each coordinate, not a variance.
spread = 0.45
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0, -1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], 0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([-1.0,  1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0, -1.0])
], 0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()

### Keep a bookmark in the random-number sequence

PyTorch generates **pseudorandom** numbers: its internal state determines the next
values it produces. `t.manual_seed(1)` set the starting state. The calls to
`t.randn(...)` above have **already generated our four clouds and advanced the
generator**.

Think of the current state as a **bookmark** in the random-number sequence. The
line below saves that bookmark **after generating the data and before generating
the network's weights and biases**. `t.get_rng_state()` reads the current state,
and `.clone()` keeps an independent copy. Saving it does not draw any new numbers.

In Exercise 1, `t.set_rng_state(parameter_initial_state)` will return the generator
to this bookmark. Repeating the following `t.randn(...)` calls in the same order
and with the same dimensions will then produce the same initial weights and biases,
even if we rerun that exercise. Restoring the bookmark affects only future random
draws: it leaves the existing dataset `X` and labels `y` unchanged. This saved state
is not a copy of the network parameters; we will create and save those later.

In [ ]:
# Supplied: save the bookmark reached after generating the data.
parameter_initial_state = t.get_rng_state().clone()

## 2. Reuse the linear classifier from Lab 4

Look at the four clouds. **Could a single straight decision boundary separate the four centres according to their labels?** Keep your prediction in mind while running the model.

The code below supplies the completed calculations from Lab 4: the stable empirical logistic loss, its gradient and the number of classification mistakes. You do not need to implement them again. Recall why we use `logsigmoid` to evaluate the logarithms directly.

As in Lab 4, we include the intercept using a column of ones. Here that design matrix is called `X_affine`, while `X` keeps the original coordinates for the later network. The functions are prefixed with `affine_` to keep the two calculations separate. Their mathematical definitions are unchanged.

The coefficient column contains $(b,w_1,w_2)$, and the scores are `X_affine @ theta`. The predicted label is 1 when the sigmoid of the score is greater than 0.5, and 0 otherwise, including at equality. This is equivalent to testing whether the score is strictly positive.

In [ ]:
# Supplied: the completed functions from Lab 4, with affine-specific names.
X_affine = t.cat([t.ones(N, 1), X], dim=1)  # Shape (N, 3), including the intercept.

def affine_loss(theta):
    scores = X_affine @ theta
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def affine_grad(theta):
    scores = X_affine @ theta
    probabilities = t.sigmoid(scores)
    return X_affine.mT @ (probabilities - y)

def affine_mistakes(theta):
    probabilities = t.sigmoid(X_affine @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return int((predicted_labels != y).sum().item())

### 2.1. Run gradient descent

Run the supplied update loop. We use Lab 4's zero initialisation and learning rate `0.01`, now allowing **1,000 updates**. Every update uses the full dataset.

As before, record the initial loss and mistake count, then record them again **after** each update. Entry `step` in each history therefore describes exactly `step` completed updates. We also store the parameter tensors at selected steps for the boundary plot.


**Recording results.** A Python list such as `[first_value]` starts with one entry;
`history.append(value)` adds another at the end. `history[0]` selects the first
entry, and `history[-1]` the last. Here entry `step` records that many completed
updates. `range(1, num_updates + 1)` includes 1 through `num_updates`, because the
upper endpoint of `range` is excluded. `.item()` extracts an ordinary Python
number from a tensor containing one value.

`affine_checkpoints = {}` creates an empty **dictionary**, which stores values
under keys. `affine_checkpoints[step] = affine_theta` stores the current parameter
tensor under the update number `step`. Each update creates a new tensor and
assigns `affine_theta` to it, so later updates leave the saved tensors unchanged.
No copy is needed here. Later, `affine_checkpoints[100]` retrieves the tensor
saved after 100 updates; it does not select a matrix row.
The condition `step in (...)` checks whether `step` is one of the listed numbers.

The printed table uses **f-strings**: in `f'Update {step}'`, Python inserts the
value of `step` into the text. A format such as `{value:.3f}` displays three
decimal places; table-width and precision settings only affect the display.

In [ ]:
affine_theta = t.zeros(3, 1)
affine_learning_rate = 0.01
num_updates = 1000
affine_loss_history = [affine_loss(affine_theta).item()]
affine_error_history = [affine_mistakes(affine_theta)]
affine_checkpoints = {}

for step in range(1, num_updates + 1):
    affine_theta = affine_theta - affine_learning_rate * affine_grad(affine_theta)
    affine_loss_history.append(affine_loss(affine_theta).item())
    affine_error_history.append(affine_mistakes(affine_theta))

    # Supplied: save this tensor; each later update creates a new one.
    if step in (1, 10, 100, num_updates):
        affine_checkpoints[step] = affine_theta

print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {affine_loss_history[step]:23.6f}   {affine_error_history[step]:8d}')

### 2.2. Inspect the curves and the boundaries

Run the supplied plotting cells. These are the same quantities we examined in Lab 4:

- the summed empirical logistic loss;
- the loss divided by $\log 2$, alongside the number of mistakes;
- the decision boundaries at successive updates.

The division by $\log 2$ is only for display: **the updates use the original, unscaled loss**. Each mistake contributes at least $\log 2$ to that loss, so the scaled loss bounds the mistake count from above.

All affine boundaries have the form $b+w_1x_1+w_2x_2=0$. For these saved coefficients, $w_2$ is nonzero, so we can draw the lines using $x_2=-(b+w_1x_1)/w_2$, just as in Lab 4. This plotting code is supplied; focus on interpreting its output.


**Drawing several panels.** Earlier labs used `plt.plot(...)` on one plot.
`fig, axes = plt.subplots(1, 2, ...)` creates a figure containing one row of two
panels and assigns the two returned objects to `fig` and `axes`. `axes[0]` is the
left panel and `axes[1]` the right. To choose a panel, use its methods:

| Command | Purpose |
|---|---|
| `axes[0].plot(horizontal_values, vertical_values, label='Model A')` | Draw a curve on the first panel. Both sequences must have the same length. |
| `axes[0].set_xlabel('Updates')` / `axes[0].set_ylabel('Loss')` | Label that panel's axes. |
| `axes[0].set_title('A comparison')` | Give that panel a title. |
| `axes[0].legend()` | Display the labels of its curves. |
| `axes[0].set_ylim(lower, upper)` | Set its vertical limits. |

`color='tab:purple'` chooses a colour; `linestyle='--'` chooses a dashed line.
Passing `'--'` as the third argument of `.plot` is the same line-style shortcut.
`figsize` sets the figure size, `constrained_layout=True` leaves room for labels,
and `.grid(alpha=0.2)` adds a faint grid. `plt.show()` displays the figure.
A loop `for ax in axes:` applies its indented instructions to each panel.

To scale every value in a list, `[value / c for value in values]` constructs a
new list by dividing each entry by `c`; the original list is unchanged. This is
called a **list comprehension**, and could also be written using a `for` loop
and `.append(...)`. `math.log(2)` is the natural logarithm of 2. In plot labels,
`$...$` asks Matplotlib to display mathematics; the `r` in `r'...'` preserves
backslashes such as the one in `\log`.

In [ ]:
import math

updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, color='0.35')
axes[0].set_title('Affine model: empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[0].set_ylim(0, 75)  # Keep the small change in perspective; the table gives exact values.

axes[1].plot(updates, [value / math.log(2) for value in affine_loss_history],
             color='tab:purple', label=r'Loss / $\log 2$')
axes[1].plot(updates, affine_error_history, color='tab:green', label='Mistakes')
axes[1].set_title('Affine model: loss and mistakes')
axes[1].set_ylabel('Value')
axes[1].set_ylim(0, 110)
axes[1].legend()
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
plt.show()

In [ ]:
# The line-plotting calculation from Lab 4, using the new observations.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_line = t.tensor(x1_limits)

fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
plt.scatter(x_class_0[:, 0], x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x_class_1[:, 0], x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
for step, style in zip((1, 10, 100, num_updates), (':', '--', '-.', '-')):
    b, w1, w2 = affine_checkpoints[step][:, 0]
    x2_line = -(b + w1 * x1_line) / w2
    plt.plot(x1_line, x2_line, color='black', linestyle=style,
             label=f'After {step} update' + ('s' if step != 1 else ''))
plt.xlim(x1_limits)
plt.ylim(x2_limits)
plt.gca().set_aspect('equal')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Affine decision boundaries')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

### Pause and interpret: what has gradient descent achieved?

1. Compare the initial and final empirical losses and mistake counts. Has the affine model fitted the observations well? The loss curve looks almost flat: does this mean that no updates have taken place? Use the printed values to read changes that are too small to see in the curves.
2. What happens to the boundary between updates 1, 100 and 1,000? Could more updates turn it into a boundary with several segments?
3. Why do the loss and the number of mistakes not necessarily decrease together?

Write down your answers before opening the explanation.

<details>
<summary>Show answer — interpreting the affine fit</summary>

**1. The loss decreases very little, and many mistakes remain.** With the supplied seed, the loss changes from approximately $69.3147$ to $69.3083$, while the mistake count changes from 50 to 47. The loss has already essentially stopped changing by update 100. The fitted model still misclassifies almost half of these observations.

The curve looks flat because the total decrease is only about $0.00645$. Initially, all coefficients are zero, so every score is zero and every predicted probability is $1/2$. The summed empirical loss is therefore $N\log 2\approx69.3147$.

For this nearly symmetric XOR sample, the fitted affine scores remain close to zero: a single straight decision boundary cannot capture the main XOR pattern. Small sample asymmetries allow a slight improvement, including a few changed class predictions, but the probabilities remain near $1/2$. The empirical logistic loss is convex in the affine coefficients, so this behaviour is not due to a suboptimal local minimum. It illustrates a limitation of the affine model on these data.

**2. Updates change the coefficients of a straight line.** The boundary initially rotates and shifts; the lines at 100 and 1,000 updates almost coincide. Further updates can change its slope and intercept, but an affine score always has a straight boundary when its feature weights are nonzero. A single straight line cannot separate the four cloud centres according to their XOR labels. We need a richer family of scores to express a different shape.

**3. The two quantities measure different things.** The logistic loss depends on the probabilities assigned to the observed labels. The mistake count only checks which side of the classification threshold each observation falls on. Probabilities can change without changing any labels, and an improvement in total loss can occur even when the number of mistakes increases.

The 47 mistakes are the outcome of fitting the logistic loss in this run. They are not a claim that 47 is the smallest mistake count attainable by every possible straight boundary.

</details>

## 3. Can we do better with a neural network?

Keep the **same observations, labels and empirical logistic loss**, and change the
model that produces the scores. In the lectures, the hidden weights and biases
made the feature functions adjustable. You will now implement that model and fit
all of its parameters.

In Lab 4, we wrote the gradient formula ourselves. Here, PyTorch's **automatic
differentiation** will calculate the derivatives for us. You will still implement
the gradient-descent updates. The next lecture explains **backpropagation**, the
algorithm behind this derivative calculation.

### Reuse the empirical logistic loss

Our Lab 4 loss function calculated the affine scores internally. The supplied
`loss_from_scores` instead takes already-computed scores and evaluates the same
empirical loss, using the labels in `y`:

$$\mathcal L_{\mathrm{log}} = \sum_{i=1}^N
\ell_{\mathrm{log}}(\sigma(s_i),y_i).$$

We can use it with either affine or neural-network scores. Recall Lab 4's numerical
stability issue: we again use `logsigmoid` to evaluate the logarithms directly.
**Pass scores to this function without first applying a sigmoid.** You do not need
to implement this familiar loss again.

In [ ]:
def loss_from_scores(scores):
    # scores contains classification scores, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

### 3.1. From the architecture to parameter tensors

Start with **two input coordinates, two hidden neurons and one scalar output**,
matching the architecture of the lecture's XOR example. In the notes' notation,
$L=2$, $d_0=2$, $d_1=2$ and $d_2=1$. Here $N=100$ counts observations;
$d_1$ counts hidden neurons and will change later. For one input row,

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,\\
\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2.
\end{aligned}
$$

The subscript identifies the layer. The column $\boldsymbol\theta$ collects all
weights and biases mathematically; our code will store them in four tensors.
`X` has shape `(N, 2)` and contains only the original coordinates. Unlike
`X_affine`, it needs no extra column of ones: biases are stored separately.

`t.randn(rows, columns, requires_grad=True)` creates a matrix of random initial
values and asks PyTorch to track computations involving it. This will let us
differentiate a loss with respect to its entries later; it does not compute any
derivatives yet. With one size argument, `t.randn(size, requires_grad=True)`
creates a one-dimensional tensor. We use one-dimensional tensors for the biases:
adding a bias vector to a matrix adds it to each row (**broadcasting**).

Recall that `A.shape` gives a tensor's dimensions. You can count its scalar
entries from those dimensions, or use `A.numel()` to obtain that count directly.
Use separate `print(...)` statements if helpful; no printing loop is required.

**Exercise 1 — initialise the parameters.**

1. Work out the shapes of `W1`, `b1`, `W2` and `b2` from the matrix products above.
2. Create them using `t.randn(..., requires_grad=True)`. Use the dimension variables
   supplied below, and create them in the order **`b1`, `W1`, `b2`, `W2`** so that
   your random draws match the reference experiment.
3. Print their shapes. How many adjustable scalar parameters are there altogether?

The supplied first line returns the random generator to the bookmark saved after
creating the data, so rerunning this cell gives the same initial parameters.

We also save the initial parameter values so that we can restore them after
fitting. For example, `initial_W1 = W1.detach().clone()` takes a snapshot of `W1`:

- `W1.detach()` returns a tensor with the same values, disconnected from automatic
  differentiation and with `requires_grad=False`. It does not change `W1`, which
  still requires gradients. The two tensors initially share the same storage.
- `.clone()` then copies those values into separate storage. Later updates to
  `W1` therefore do not change the saved values in `initial_W1`.

The saved tensor is only a record of the starting values; we do not need derivatives
for it. The supplied final lines do this for all four parameter tensors. Keep them
in this cell, below your initialisations.

In [ ]:
t.set_rng_state(parameter_initial_state)  # Supplied: return to the bookmark before drawing parameters.
input_features = 2
hidden_features = 2
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2, then print shapes and count scalar parameters.

# Supplied: save the initial values immediately after creating the parameters.
initial_b1 = b1.detach().clone()
initial_W1 = W1.detach().clone()
initial_b2 = b2.detach().clone()
initial_W2 = W2.detach().clone()

<details>
<summary>Show answer — parameter shapes and initialisation</summary>

`W1` has shape `(2, 2)`, `b1` has shape `(2,)`, `W2` has shape
`(2, 1)` and `b2` has shape `(1,)`. The total is
$2\times2+2+2\times1+1=9$ scalar parameters.
The notes' bias rows are represented by one-dimensional tensors in this code.

```python
t.set_rng_state(parameter_initial_state)  # Supplied: return to the bookmark before drawing parameters.
input_features = 2
hidden_features = 2
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)

print('W1:', W1.shape)
print('b1:', b1.shape)
print('W2:', W2.shape)
print('b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())

# Supplied: save the initial values immediately after creating the parameters.
initial_b1 = b1.detach().clone()
initial_W1 = W1.detach().clone()
initial_b2 = b2.detach().clone()
initial_W2 = W2.detach().clone()
```

</details>

### 3.2. Implement the network calculation

The equations above describe one observation. Applying them to the whole matrix
`X` performs the same calculation for every row, using the same parameters.
`t.relu(A)` applies $\max\{0,a\}$ separately to every entry of `A`; it implements
the componentwise ReLU from the notes.

**Exercise 2 — write `net(X)`.** Translate the three equations into a Python
function with intermediate variables `a1` and `phi1`, returning `scores`.
Use the four tensors you have just created; do not draw new parameters inside
the function. Before running it, predict the shapes of all three intermediate
results for the dataset. What should happen if you pass only one input row as
`X[:1]`? This slice keeps a matrix with shape `(1, 2)`; `X[0]` would instead
return a one-dimensional tensor.

Use `@` for matrix multiplication and keep these calculations as PyTorch tensors.
The function must return its scores, not just print them. We will differentiate
this calculation later, so do not use `.item()` or `detach()` inside `net`.

The supplied checks after your implementation use `assert condition`: if the
condition is false, execution stops with an `AssertionError`. These checks test
the output shapes; they do not establish that every part of an implementation
is correct. If a check fails, inspect the intermediate shapes and formulas.

The hidden activations are values of learned feature functions. The final output
is a classification **score**, not a probability or a predicted label.

In [ ]:
# YOUR CODE: define net(X).

<details>
<summary>Show answer — the network calculation</summary>

`a1` and `phi1` have shape `(N, 2)`; `scores` has shape `(N, 1)`.
For a single-row input, `a1` and `phi1` have shape `(1, 2)`, and `scores` has
shape `(1, 1)`. The variables contain values for the inputs passed to the
function; the parameter tensors are reused.

```python
def net(X):
    a1 = X @ W1 + b1
    phi1 = t.relu(a1)
    scores = phi1 @ W2 + b2
    return scores
```

</details>

In [ ]:
# Supplied checks: one score per observation, also for a single-row input.
assert net(X).shape == y.shape
assert net(X[:1]).shape == (1, 1)
print('Dataset output shape:', net(X).shape)

### 3.3. Evaluate the empirical loss and mistakes

Recall the operations used to count mistakes in Lab 4:

- Comparisons such as `A > 0` and `A != B` act entry by entry and return tensors
  of `True`/`False` values. `!=` means "not equal".
- Summing a Boolean tensor with `.sum()` counts its `True` entries.
- `A.dtype` describes the type of its entries; `B.to(A.dtype)` converts `B` to
  that type. This lets Boolean predictions be represented numerically as 0 and 1.
- `.item()` extracts a Python number from a one-value tensor, and `int(...)`
  converts a number to a Python integer.

Keep predicted labels and observed labels at the same shape `(N, 1)` when you
compare them. Work out which comparison identifies an incorrect prediction.

**Exercise 3 — evaluate your initial model.**

1. Calculate its scores and empirical logistic loss using `loss_from_scores`.
2. Write `mistakes_from_scores(scores)`, returning the number of incorrect labels
   as a Python integer. Like the loss helper, it should use the fixed labels `y`.
   Reuse Lab 4's decision rule: class 1 for a strictly positive score and class 0
   otherwise, including when the score is zero.
3. Print the loss and the number of mistakes before fitting.

Do not add a sigmoid to the network output for the loss.
To repeat this initial evaluation after fitting, rerun Exercise 1 first.

In [ ]:
# YOUR CODE: define mistakes_from_scores, then evaluate the initial model.

<details>
<summary>Show answer — initial loss and mistakes</summary>

Compare each predicted label with its observed label and count disagreements.
The supplied initialisation gives loss about $75.171$ and 56 mistakes. These are
the randomly initialised model's results before any fitting updates.

```python
def mistakes_from_scores(scores):
    predicted_labels = (scores > 0).to(y.dtype)
    return int((predicted_labels != y).sum().item())

scores = net(X)
L = loss_from_scores(scores)
print('Initial empirical loss:', L.item())
print('Initial mistakes:', mistakes_from_scores(scores))
```

</details>

In [ ]:
# Supplied checks on artificial scores, including ties at zero.
assert mistakes_from_scores(t.zeros_like(y)) == 50
assert mistakes_from_scores(2 * y - 1) == 0
print('Mistake-count checks passed.')

### 3.4. Ask PyTorch for the derivatives

Here is a small example separate from the network. Predict the two partial
derivatives of $Q(u)=u_0^2+u_1^2$ at $u=(1,-2)$, then run the cell.
`Q.backward()` computes the derivatives and stores them in `u.grad`.

In [ ]:
u = t.tensor([1.0, -2.0], requires_grad=True)
Q = (u ** 2).sum()
Q.backward()
print('u:', u)
print('Gradient of Q:', u.grad)

The gradient is $(2,-4)$ and `u` has not changed. For a network parameter tensor,
each `.grad` entry is similarly the derivative of the scalar empirical loss with
respect to the parameter entry in the same position. For example, `W1.grad[0, 0]`
corresponds to `W1[0, 0]`. Initially, `.grad` is `None`, Python's special value
for an absent value. Here it means that no gradient tensor has been stored yet.
This is different from a stored gradient whose entries are all zero.

**`backward()` does not update parameters. It adds to any gradients already stored.**
The gradient of a sum is the sum of the gradients: we could calculate losses for
separate observations and call `backward()` for each, holding parameters fixed.
The accumulated result would be the gradient needed for one full-dataset update.
This is useful when data must be processed in smaller pieces.

Our empirical loss already sums over the whole small dataset using tensor
operations. One call computes its full gradient, without a Python loop over
observations. We therefore clear old gradients before calculating the next ones.
**Calculating a new loss does not clear `.grad`.** PyTorch uses 0 as the ReLU
multiplier at input 0; its ordinary derivative remains undefined there.

**How the supplied clearing code works.**

- `for parameter in (b1, W1, b2, W2):` runs the indented block once for each of
  the four tensors. On each turn, `parameter` refers to that tensor.
- `if parameter.grad is not None:` runs its indented line only if a gradient
  tensor is already stored. When `.grad` is `None`, there is nothing to clear.
- `parameter.grad.zero_()` fills that gradient tensor with zeros. The trailing
  underscore indicates that it changes the tensor in place. **It clears the
  stored derivatives without changing the parameter values.**

After initialising the parameters, the first run skips the clearing operation.
If you run this exercise again, the same code clears the previously calculated
gradients before you compute new ones. Without the check, trying to call
`.zero_()` on `None` would give an error. This setup is supplied; you do not need
to write it yourself.

**Exercise 4 — compute the network's gradients.** After the supplied clearing
code, calculate a fresh loss from your network and ask PyTorch to differentiate
it. Print each parameter's shape and its gradient's shape. You can use a separate
`print(...)` for each parameter, comparing, for example, `W1.shape` with
`W1.grad.shape`.

The derivatives refer to the current parameter values. If you want to examine
the initial parameters again after fitting, rerun Exercise 1 first.

Explain why the shapes match. If an entry of `W1.grad` is positive, should the
corresponding entry of `W1` increase or decrease in a gradient-descent update
with a positive learning rate?

In [ ]:
# Supplied: make this gradient demonstration repeatable.
for parameter in (b1, W1, b2, W2):  # Visit each parameter tensor.
    if parameter.grad is not None:  # Only clear a gradient if one is stored.
        parameter.grad.zero_()     # Clear derivatives, not parameter values.

# YOUR CODE: calculate a fresh loss, differentiate it, and print the shapes.

<details>
<summary>Show answer — derivatives of the network loss</summary>

Each loss derivative corresponds to one parameter entry, so the shapes agree.
A positive derivative gives a negative change in that weight because gradient
descent subtracts the learning rate times the derivative. All these derivatives
are evaluated before any parameters are changed.

```python
# Supplied: make this gradient demonstration repeatable.
for parameter in (b1, W1, b2, W2):  # Visit each parameter tensor.
    if parameter.grad is not None:  # Only clear a gradient if one is stored.
        parameter.grad.zero_()     # Clear derivatives, not parameter values.

L = loss_from_scores(net(X))
L.backward()
print('W1:', W1.shape, W1.grad.shape)
print('b1:', b1.shape, b1.grad.shape)
print('W2:', W2.shape, W2.grad.shape)
print('b2:', b2.shape, b2.grad.shape)
```

</details>

### 3.5. Implement gradient descent

You already know the update rule: subtract the learning rate times each current
partial derivative from the corresponding parameter. Two pieces of Python/PyTorch
syntax let us apply it to tensors that require gradients.

**`with` is a Python keyword introducing a block with a temporary context.**
`with t.no_grad():` turns off gradient tracking for the indented instructions.
They run normally; after the block, the previous tracking setting is restored.
We have already computed the derivatives and do not need to differentiate the
parameter-update operation itself. This context does not clear stored gradients.

Calculate the fresh loss that you intend to differentiate, and call `backward()`,
**outside** the `no_grad` block. PyTorch must record that loss calculation to
obtain its derivatives. Use `no_grad` for updates and for measurements that you
will not differentiate.

**`.add_(...)` changes a tensor in place.** For example, the following cell applies
one update to the separate toy variable `u` above. Then `u.grad.zero_()` clears
the stored derivatives without changing `u`. Run the toy derivative cell again
first if you want to repeat this example from its initial values.

For our parameter tensors, use these in-place updates rather than assigning a
new tensor to each parameter name. This keeps the same parameter tensors, still
requiring gradients, available for the next derivative calculation.

In [ ]:
with t.no_grad():
    u.add_(-0.1 * u.grad)
u.grad.zero_()
print('After the toy update:', u)  # (0.8, -1.6)
print('Cleared gradient:', u.grad)

The next cell supplies the experiment's reset and plotting setup. Run it before
the fitting exercise. `restart_network()` restores the initial parameter values
saved in Exercise 1 and clears any stored gradients. `.copy_(...)` copies values
into an existing tensor in place. It does not draw a new initialisation.

The grid is only for pictures of decision boundaries. It contains no extra
observations used for fitting and does not alter the empirical loss.

The supplied code constructs the picture coordinates: `linspace` chooses evenly
spaced values, `meshgrid` forms two rectangular coordinate arrays, and `stack`
pairs their coordinates. `X_grid` stores those pairs as rows so `net` can process
them like any other input matrix. `.reshape(-1, 2)` keeps two columns and lets
PyTorch infer the number of rows. Reshaping scores to `x1_grid.shape` puts them
back into the picture's rectangular arrangement without changing their values.
The `*` in `t.linspace(*x1_limits, 250)` passes the two entries of `x1_limits` as
the separate start and end arguments. You do not need to reproduce this setup.

In [ ]:
# Supplied experiment infrastructure.
def restart_network():
    with t.no_grad():
        b1.copy_(initial_b1)
        W1.copy_(initial_W1)
        b2.copy_(initial_b2)
        W2.copy_(initial_W2)
    for parameter in (b1, W1, b2, W2):
        if parameter.grad is not None:
            parameter.grad.zero_()

# Supplied plotting grid; reproducing this construction is not an exercise.
x1_limits = (X[:, 0].min().item() - 0.5, X[:, 0].max().item() + 0.5)
x2_limits = (X[:, 1].min().item() - 0.5, X[:, 1].max().item() + 0.5)
x1_grid, x2_grid = t.meshgrid(t.linspace(*x1_limits, 250),
                            t.linspace(*x2_limits, 250), indexing='xy')
X_grid = t.stack([x1_grid, x2_grid], dim=-1).reshape(-1, 2)

**Exercise 5 — adapt gradient descent to all four parameter tensors.**

Complete the marked parts of the cell below. Use the full dataset for each of
1,000 updates, with learning rate `0.001`. We will keep these settings when we
widen the network and add another hidden layer later. We use a different learning rate from
the affine model because its parameterisation differs; we are comparing the
resulting fits, not measuring which algorithm needs fewer updates.

1. Create `loss_history` and `error_history`, each starting with the initial
   measurement. Store ordinary Python numbers, not tensors with tracked computations.
2. At each step, compute a fresh empirical loss and all its gradients.
3. Inside the supplied `no_grad` block, update **all four** parameter tensors using
   those gradients. Then recompute scores and append the new loss and mistake count.
4. Clear the stored gradients before the next iteration.

Reuse the list operations from Section 2: initialise each history with one
measurement and use `.append(...)` for each later one. Extract loss values with
`.item()`; `mistakes_from_scores` already returns a Python integer. Initial
measurements can be calculated inside `no_grad`, as they will not be differentiated.

All derivatives must be calculated at the same current parameters, before any
parameters are updated. Entry `step` in each history must describe exactly
`step` completed updates, so each finished history has 1,001 entries.

The reset at the start makes reruns repeat the experiment. Keep the supplied
snapshot lines where they are: they store scores **after** the selected updates,
before later updates change the network again. The snapshots are only for plots.

As with `affine_checkpoints`, `boundary_scores` is a dictionary. Its keys are
completed update numbers, and its values are saved matrices of scores on the
plotting grid. For example, `boundary_scores[100]` retrieves the snapshot saved
after 100 updates. The supplied `.clone()` gives each snapshot its own storage.

In [ ]:
restart_network()  # Supplied: every run begins at the saved initial parameters.
learning_rate = 0.001
num_updates = 1000
boundary_steps = (1, 100, num_updates)
boundary_scores = {}

# YOUR CODE: initialise loss_history and error_history, including update 0.

for step in range(1, num_updates + 1):
    # YOUR CODE: evaluate a fresh loss and compute its gradients.

    with t.no_grad():
        # YOUR CODE: update b1, W1, b2 and W2 using their gradients.
        # YOUR CODE: recompute scores and append loss and mistakes to the histories.

        # Supplied: save independent score values for the later boundary figures.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    # YOUR CODE: clear the stored gradients before the next iteration.

<details>
<summary>Show answer — the gradient-descent loop</summary>

The algorithm is the same as in Lab 4. The parameters now occupy four tensors,
and `backward()` calculates their derivatives. Measurements below use the
updated parameters; the next iteration constructs a fresh loss for differentiation.
We clear all four gradients, not the parameters themselves.

```python
restart_network()  # Supplied: every run begins at the saved initial parameters.
learning_rate = 0.001
num_updates = 1000
boundary_steps = (1, 100, num_updates)
boundary_scores = {}

with t.no_grad():
    scores = net(X)
    loss_history = [loss_from_scores(scores).item()]
    error_history = [mistakes_from_scores(scores)]

for step in range(1, num_updates + 1):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

        scores = net(X)
        loss_history.append(loss_from_scores(scores).item())
        error_history.append(mistakes_from_scores(scores))

        # Supplied: save independent score values for the later boundary figures.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    b1.grad.zero_()
    W1.grad.zero_()
    b2.grad.zero_()
    W2.grad.zero_()
```

</details>

In [ ]:
# Supplied checks and summary of your recorded results.
assert len(loss_history) == num_updates + 1
assert len(error_history) == num_updates + 1
assert set(boundary_scores) == set(boundary_steps)
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {loss_history[step]:23.6f}   {error_history[step]:8d}')

Run the supplied cell below immediately after fitting the two-neuron network.
It saves independent copies of its histories and final grid scores for the later
width comparison. The width check prevents accidentally replacing these results
with those of the larger network. If you return here after widening the network,
first rerun the two-neuron initialisation and fitting exercises.

For a list of numerical measurements, `.copy()` makes a separate list, so later
changes to the working history will not alter the saved history. For the grid
scores we use `detach().clone()`, as in the parameter snapshots in Exercise 1.
The supplied check `t.allclose(A, B)` returns `True` when corresponding entries
agree within numerical tolerances; it allows small floating-point rounding errors.

In [ ]:
# Supplied: preserve this completed two-neuron experiment before changing width.
assert W1.shape == (2, 2), 'Run the two-neuron initialisation and fit first.'
assert len(loss_history) == len(error_history) == num_updates + 1
assert num_updates == 1000 and learning_rate == 0.001
with t.no_grad():
    assert abs(loss_from_scores(net(X)).item() - loss_history[-1]) < 1e-4
    assert mistakes_from_scores(net(X)) == error_history[-1]
    assert t.allclose(net(X_grid).reshape(x1_grid.shape), boundary_scores[num_updates])
narrow_loss_history = loss_history.copy()
narrow_error_history = error_history.copy()
narrow_grid_scores = boundary_scores[num_updates].detach().clone()

## 4. Examine what your network learned

### 4.1. Plot the loss and mistakes

**Exercise 6 — add two diagnostic curves.** The supplied code prepares three
panels and draws the affine curves and the network's scaled loss:

- Left: empirical logistic loss against completed updates, for both models.
- Middle: affine loss divided by $\log 2$, alongside its mistake count.
- Right: the same two quantities for your neural network.

**Write only two `plot` calls between the markers.** Add the unscaled network
loss on `axes[0]`, using `updates` and `loss_history`; add network mistakes on
`axes[2]`, using `updates` and `error_history`. Adapt the supplied affine calls:
use `color='tab:purple', label='Two-neuron network'` for loss and
`color='tab:green', label='Mistakes'` for errors. All formatting and scaling
below are supplied; you do not need to reconstruct them.

The scaling is for display only: each mistake contributes at least $\log 2$ to
the summed logistic loss, so mistakes cannot exceed loss divided by $\log 2$.
**The fitting updates used the unscaled empirical loss.**

In [ ]:
# Supplied: layout, affine comparison and scaled-loss curves.
updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
axes[0].plot(updates, affine_loss_history, '--', color='0.4', label='Affine score')
scaled_affine_loss = t.tensor(affine_loss_history) / math.log(2)
scaled_network_loss = t.tensor(loss_history) / math.log(2)
axes[1].plot(updates, scaled_affine_loss, color='tab:purple', label=r'Loss / $\log 2$')
axes[1].plot(updates, affine_error_history, color='tab:green', label='Mistakes')
axes[2].plot(updates, scaled_network_loss, color='tab:purple', label=r'Loss / $\log 2$')

# YOUR CODE START
# Add network loss on axes[0] and network mistakes on axes[2].
# YOUR CODE END

# Supplied: titles, labels and display.
axes[0].set_title('Empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[1].set_title('Affine model: loss and mistakes')
axes[1].set_ylabel('Value')
axes[1].set_ylim(0, 120)
axes[2].set_title('Two-neuron network: loss and mistakes')
axes[2].set_ylabel('Value')
axes[2].set_ylim(0, 120)
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()

<details>
<summary>Show answer — the two missing curves</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

The horizontal coordinates count completed updates, including update 0.
The two selected histories contain the network's loss and mistake count.

```python
axes[0].plot(updates, loss_history, color='tab:purple', label='Two-neuron network')
axes[2].plot(updates, error_history, color='tab:green', label='Mistakes')
```

</details>

### 4.2. Inspect the decision boundaries

A neural network's zero-score set need not be a single straight line. The supplied
helper below draws it from scores already evaluated on a grid, tracing the zero
contour.
The black line separates light blue class-0 regions from light orange class-1
regions. At exactly zero, our classifier selects class 0.

Run this helper definition; reproducing its contour code is not an exercise,
here or in the later comparisons. You will only call the supplied function.
`draw_boundary(ax, score_grid, title)` needs the axes on which to draw, the matrix
of grid scores, and a title. You will choose which saved network scores it displays.

In [ ]:
def draw_boundary(ax, score_grid, title):
    # Non-examinable helper: shade predicted classes and draw the score = 0 contour.
    classes = (score_grid > 0).to(t.float32)
    ax.contourf(x1_grid, x2_grid, classes,
                levels=[-0.5, 0.5, 1.5], colors=['#edf3fb', '#fff1e9'])
    ax.contour(x1_grid, x2_grid, score_grid, levels=[0], colors='black', linewidths=1)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    ax.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', s=18, label='Class 0')
    ax.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', s=20, label='Class 1')
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$', title=title)
    ax.set_aspect('equal')

**Exercise 7 — adapt a supplied boundary plot.**

The cell below creates four panels. We supply the affine model's plot and the
neural network's plot after one update. **Run the cell as supplied first**: the
first two panels will contain plots and the last two will be empty.

Here is how the supplied neural-network call works:

```python
draw_boundary(axes[1], boundary_scores[1],
              f'2-neuron NN: 1 update\nMistakes: {error_history[1]}')
```

- `axes[1]` chooses the **second panel**: Python counts from zero, so `axes[0]`
  is the first. This index selects where to draw, not the update number.
- `boundary_scores[1]` contains the scores saved on the plotting grid after
  **one update**. The helper uses their signs for the shaded classes and traces
  the zero-score boundary. Pass these saved scores, not probabilities or labels.
- The final argument is the title. In an `f'...'` string, `{error_history[1]}`
  inserts the mistake count after one update. The escape `\n` starts a new line.
  The counts refer to the observed dataset; the grid is only used to draw the picture.

Now add two calls **between the markers**: use the third and fourth panels for
100 and 1,000 updates. Adapt the supplied neural-network call by changing the
panel, the saved scores, and the update number and mistake count in the title.
Use separate calls; no loop or changes to the helper are needed. Rerun the whole
cell to display all four panels. Exercises 11 and 15 reuse exactly this pattern
with the saved scores for the other networks.

Why would calling `net(X_grid)` three times here fail to recover those three
earlier stages? Which array holds the mistake count for each completed update?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
affine_grid = t.cat([t.ones(X_grid.shape[0], 1), X_grid], dim=1)
affine_grid_scores = (affine_grid @ affine_theta).reshape(x1_grid.shape)
# Supplied: the affine comparison and one complete neural-network example.
draw_boundary(axes[0], affine_grid_scores,
              f'Affine: {num_updates} updates\nMistakes: {affine_error_history[-1]}')
draw_boundary(axes[1], boundary_scores[1],
              f'2-neuron NN: 1 update\nMistakes: {error_history[1]}')
# YOUR CODE START
# Adapt the NN call for 100 updates on axes[2] and 1000 on axes[3].
# YOUR CODE END
axes[0].legend(loc='upper left', fontsize=8)
plt.show()

<details>
<summary>Show answer — the two missing boundary calls</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

The parameters currently contain the final fit: calling `net` again would show
that final model every time. `boundary_scores` holds the independently saved
values at each selected update; `error_history[step]` holds its mistake count.

```python
draw_boundary(axes[2], boundary_scores[100],
              f'2-neuron NN: 100 updates\nMistakes: {error_history[100]}')
draw_boundary(axes[3], boundary_scores[num_updates],
              f'2-neuron NN: {num_updates} updates\nMistakes: {error_history[num_updates]}')
```

</details>

### 4.3. Check that the feature values changed

**Exercise 8 — inspect the learned representation of one fixed input.**

Keep the input row `x = X[:1]` fixed. Using the hidden-layer formula from Section
3.1, compute its hidden activations before fitting (from `initial_W1` and
`initial_b1`) and after fitting (from the current `W1` and `b1`). Use `no_grad`
because this is an inspection, not a derivative calculation.

Print both entries of each feature row and the largest absolute change between
them. `t.abs(A)` takes absolute values entry by entry, and `A.max()` returns the
largest entry as a scalar tensor; `.item()` extracts its numerical value.
Did the input change? Which parameters are responsible for the changed feature values? Would
adjusting only `W2` and `b2` have changed these hidden activations?

In [ ]:
# YOUR CODE: compare the same input's hidden activations before and after fitting.

<details>
<summary>Show answer — the same input, changed features</summary>

The input row is identical in both calculations. Changing `W1` and `b1` changes
the feature functions and hence can change their values at that input. The output
parameters combine these values into scores; changing only `W2` and `b2` would
leave the hidden activations unchanged. We are inspecting one row here, not
claiming every feature changes at every input.

```python
x = X[:1]
with t.no_grad():
    features_before = t.relu(x @ initial_W1 + initial_b1)
    features_after = t.relu(x @ W1 + b1)
print('Before:', features_before[0])
print('After: ', features_after[0])
print('Largest absolute change:', t.abs(features_after - features_before).max().item())
```

</details>

### Pause and interpret: what did two hidden neurons add?

1. Compare the final empirical loss and mistake count with the affine model.
   What changed in the shape of the boundary?
2. Which parameters determine the feature functions, and which combine their
   values into a score? Did we supply the lecture's XOR features ourselves?
3. Inspect the loss and mistake counts at updates 36 and 37. Does reducing the
   loss necessarily reduce the number of mistakes?
4. We still make mistakes, although two neurons solved XOR in the lectures.
   What has changed in our data and in how we choose the parameters?

Write down your answers before opening the explanation. The next exercise
examines the difference from the lecture example directly.

<details>
<summary>Show answer — interpreting the two-neuron fit</summary>

**1. The fitted network improves on the affine score.** After 1,000 updates, the
affine model has loss about $69.308$ and makes 47 mistakes. The two-neuron network
has loss about $25.196$ and makes 10 mistakes. Its boundary can have more than one
segment, rather than being a single straight line.

**2. The hidden parameters determine the features.** `W1` and `b1` define the
feature functions; `W2` and `b2` combine their values into scores. We chose the
architecture and random initial parameters. Gradient descent adjusted all four
tensors; we did not prescribe the lecture's two feature functions.

**3. Loss and mistakes measure different things.** From update 36 to 37, the loss
falls from about $69.212$ to $69.129$, while mistakes rise from 33 to 34. The loss
uses the probabilities for every observation; mistakes only use thresholded labels.

**4. Both the data and the parameter choice have changed.** In the lecture, we
constructed parameters that classify four ideal XOR inputs correctly. Here the
coordinates are noisy and we learn parameters from a random starting point using
a finite number of updates. The remaining errors do not establish the smallest
error that any two-neuron network could achieve. We investigate the role of the
perturbed inputs next.

These values refer to the supplied initialisation, learning rate and update count.
Rerunning the fitting cell restores its initial parameters. Small numerical
differences across PyTorch versions and devices are possible.

</details>

## 5. Why did two features solve XOR in the lecture?

Our cloud centres are at coordinates $-1$ and $1$, instead of the lecture's Boolean
inputs $0$ and $1$. Rescaling the same separating construction gives the score

$$s(\boldsymbol x)=\operatorname{ReLU}(x_1-x_2)
+\operatorname{ReLU}(x_2-x_1)-1=|x_1-x_2|-1.$$

It is positive at the class-1 centres $(-1,1)$ and $(1,-1)$, and negative at the
class-0 centres $(-1,-1)$ and $(1,1)$.

**Exercise 9 — compare the ideal inputs with the noisy observations.**

1. Write `lecture_score(inputs)` for this score, returning one column of scores.
   Use slices such as `inputs[:, 0:1]` to preserve the column dimension.
2. Count its mistakes on the four supplied centres, using `centre_y`. Our helper
   `mistakes_from_scores` uses the 100 labels in `y`, so do not use it for the centres.
3. Evaluate the same score on the observed data `X`; now use `mistakes_from_scores`
   and `loss_from_scores` to report its mistakes and empirical loss.
4. Explain why the lecture's exact classification of four inputs no longer
   guarantees correct labels for every observation in the clouds. Do your results
   show that this finite sample cannot be classified perfectly by any network?

Use this separate function without changing `net` or its learned parameters.

In [ ]:
centre_X = t.tensor([[-1.0, -1.0], [1.0, 1.0], [-1.0, 1.0], [1.0, -1.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

# YOUR CODE: define lecture_score and compare the centres with the noisy data.

<details>
<summary>Show answer — ideal XOR versus noisy observations</summary>

The constructed score makes **0 mistakes on the centres**, but **8 on the noisy
observations**, with empirical loss about $43.123$. Each observation keeps its
cloud's label while its coordinates are perturbed. Some observations move across
the boundary that correctly separates the four centres. The construction therefore
does not inherit a guarantee of zero mistakes on these new inputs.

This shows the effect of the noisy coordinates on this particular classifier.
It does **not** prove that no classifier can fit all 100 observed labels. Nor can
we attribute every error of our fitted network solely to noise: its result also
depends on the available functions, initialisation and optimisation.

The fitted two-neuron network makes 10 mistakes but has a smaller logistic loss
($25.196$) than this constructed score ($43.123$). Fewer mistakes and smaller
logistic loss are different objectives. Our algorithm minimises the latter.

```python
centre_X = t.tensor([[-1.0, -1.0], [1.0, 1.0], [-1.0, 1.0], [1.0, -1.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

def lecture_score(inputs):
    difference = inputs[:, 0:1] - inputs[:, 1:2]
    return t.relu(difference) + t.relu(-difference) - 1

centre_scores = lecture_score(centre_X)
centre_mistakes = int(((centre_scores > 0).to(centre_y.dtype) != centre_y).sum().item())
noisy_scores = lecture_score(X)
print('Mistakes on the four centres:', centre_mistakes)
print('Mistakes on the noisy observations:', mistakes_from_scores(noisy_scores))
print('Empirical loss on the noisy observations:', loss_from_scores(noisy_scores).item())
```

</details>

## 6. What changes with 100 hidden neurons?

We now increase the hidden width from $d_1=2$ to $d_1=100$. **The number of
observations remains $N=100$**: we keep `X`, `y` and the same empirical loss.
We also keep learning rate `0.001` and 1,000 full-dataset updates.

**Exercise 10 — widen and fit your network.**

**Part A.** In the new cell below, copy your parameter-initialisation code from
Exercise 1 and change `hidden_features` to 100. Keep the RNG bookmark restoration,
the draw order and the lines that save the initial parameter copies. Predict the
new shapes and total parameter count first. Which tensor's shape stays unchanged?

Keep the earlier exercise cells as the two-neuron experiment. Restoring the
bookmark makes this wider initialisation repeatable; changing the dimensions means
it is not the same set of initial weights as in the smaller network.

In [ ]:
# YOUR CODE: copy your initialisation from Exercise 1, now with hidden_features = 100.
# Keep the bookmark restoration and the saved initial parameter copies.

<details>
<summary>Show answer — initialising the wider network</summary>

The parameter count increases from 9 to 401: `W1` is `(2, 100)`, `b1` is `(100,)`,
and `W2` is `(100, 1)`. `b2` remains `(1,)` because there is still one output score.
The same network formula applies with these new dimensions.

```python
t.set_rng_state(parameter_initial_state)  # Supplied: return to the bookmark before drawing parameters.
input_features = 2
hidden_features = 100
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)

print('W1:', W1.shape)
print('b1:', b1.shape)
print('W2:', W2.shape)
print('b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())

# Supplied: save the initial values immediately after creating the parameters.
initial_b1 = b1.detach().clone()
initial_W1 = W1.detach().clone()
initial_b2 = b2.detach().clone()
initial_W2 = W2.detach().clone()
```

</details>

**Part B.** Copy your completed fitting cell from Exercise 5 into the cell below
and run it. The definitions of `net`, the loss, the mistake count and
`restart_network` can all be reused: they refer to the four current parameter
tensors. Keep all four updates, gradient clearing, initial measurements and saved
boundary snapshots. Explain why the update rule needs no mathematical change.

Your narrow-network histories and final grid scores were saved separately after
its fit. This new fitting cell replaces only the working histories and snapshots.
If you later return to the narrow experiment, run its initialisation before its fit;
similarly, run Part A before refitting the wide model after switching widths.

In [ ]:
# YOUR CODE: reuse your completed gradient-descent cell from Exercise 5.
# Keep learning_rate = 0.001 and num_updates = 1000.

<details>
<summary>Show answer — the same fitting algorithm</summary>

The loop already updates every entry of each parameter tensor. Increasing the
width changes their shapes, while the score formula, differentiation call and
gradient-descent rule remain applicable. The reset at the start restores the
wide model's saved initial values, including when you repeat this cell.

```python
restart_network()  # Supplied: every run begins at the saved initial parameters.
learning_rate = 0.001
num_updates = 1000
boundary_steps = (1, 100, num_updates)
boundary_scores = {}

with t.no_grad():
    scores = net(X)
    loss_history = [loss_from_scores(scores).item()]
    error_history = [mistakes_from_scores(scores)]

for step in range(1, num_updates + 1):
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

        scores = net(X)
        loss_history.append(loss_from_scores(scores).item())
        error_history.append(mistakes_from_scores(scores))

        # Supplied: save independent score values for the later boundary figures.
        if step in boundary_steps:
            boundary_scores[step] = net(X_grid).reshape(x1_grid.shape).clone()

    b1.grad.zero_()
    W1.grad.zero_()
    b2.grad.zero_()
    W2.grad.zero_()
```

</details>

In [ ]:
# Supplied checks: keep the two experiments distinct.
assert W1.shape == (2, 100)
assert len(loss_history) == len(error_history) == 1001
assert len(narrow_loss_history) == len(narrow_error_history) == 1001
assert set(boundary_scores) == {1, 100, 1000}
print('Model             Parameters   Final loss   Mistakes')
print(f'Affine                     3   {affine_loss_history[-1]:10.3f}   {affine_error_history[-1]}')
print(f'2 hidden neurons           9   {narrow_loss_history[-1]:10.3f}   {narrow_error_history[-1]}')
print(f'100 hidden neurons       401   {loss_history[-1]:10.3f}   {error_history[-1]}')

Before the next comparison, preserve the completed 100-neuron fit too. The
supplied cell uses the same copying operations as for the two-neuron fit.
`wide_loss_history`, `wide_error_history` and `wide_grid_scores` will keep these
results available for the final comparison with a two-hidden-layer network.
If you revisit this cell after switching to two neurons, first rerun Exercise 10.

In [ ]:
# Supplied: preserve the completed 100-neuron experiment.
assert W1.shape == (2, 100), 'Run the 100-neuron initialisation and fit first.'
assert len(loss_history) == len(error_history) == num_updates + 1
assert num_updates == 1000 and learning_rate == 0.001
with t.no_grad():
    assert abs(loss_from_scores(net(X)).item() - loss_history[-1]) < 1e-4
    assert mistakes_from_scores(net(X)) == error_history[-1]
    assert t.allclose(net(X_grid).reshape(x1_grid.shape), boundary_scores[num_updates])
wide_loss_history = loss_history.copy()
wide_error_history = error_history.copy()
wide_grid_scores = boundary_scores[num_updates].detach().clone()

**Exercise 11 — compare the two widths.**

**Part A: curves.** We supply the layout and the two-neuron curves. Add just two
`plot` calls between the markers: use `wide_loss_history` on `axes[0]` and
`wide_error_history` on `axes[1]`, both against `updates`. Adapt the supplied
calls, using solid lines (omit `'--'`) and `label='100 hidden neurons'`.
Keep loss purple and mistakes green. Titles, axes and legends are supplied.

In [ ]:
# Supplied: layout and two-neuron reference curves.
updates = range(num_updates + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(updates, narrow_loss_history, '--', color='tab:purple', label='2 hidden neurons')
axes[1].plot(updates, narrow_error_history, '--', color='tab:green', label='2 hidden neurons')

# YOUR CODE START
# Add the 100-neuron loss on axes[0] and mistakes on axes[1].
# YOUR CODE END

# Supplied: formatting and display.
axes[0].set_title('Empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[1].set_title('Classification mistakes')
axes[1].set_ylabel('Mistakes')
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()

<details>
<summary>Show answer — adding the wider network</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

The saved wide histories contain the measurements from the second fit.

```python
axes[0].plot(updates, wide_loss_history, color='tab:purple', label='100 hidden neurons')
axes[1].plot(updates, wide_error_history, color='tab:green', label='100 hidden neurons')
```

</details>

**Part B: boundaries.** The first two panels below are supplied. Add one
`draw_boundary` call on `axes[2]` for the 100-neuron model, using its saved
`wide_grid_scores` and the supplied `wide_title`. This is the same function
call you used in Exercise 7; do not rewrite the helper.

Which model has the smaller empirical loss and fewer mistakes? How does the
wider network's boundary differ? Do these results guarantee a better fit for
every initialisation, or better predictions on unseen observations?

In [ ]:
# Supplied: layout, reference boundaries and title.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
draw_boundary(axes[0], affine_grid_scores, f'Affine\nMistakes: {affine_error_history[-1]}')
draw_boundary(axes[1], narrow_grid_scores, f'2 hidden neurons\nMistakes: {narrow_error_history[-1]}')
wide_title = f'100 hidden neurons\nMistakes: {wide_error_history[-1]}'

# YOUR CODE START
# Draw the saved 100-neuron boundary on axes[2], using wide_title.
# YOUR CODE END

# Supplied: display.
axes[0].legend(loc='upper left', fontsize=8)
plt.show()

<details>
<summary>Show answer — the wider boundary and interpretation</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

The two-neuron fit ends at loss about $25.196$ with 10 mistakes; the 100-neuron
fit reaches $4.713$ with 1 mistake. The wider boundary follows the clouds more
closely in this run. Changing width also changes the initial parameter arrays;
these results do not guarantee better fitting in every run, a global minimum,
or better predictions on unseen data.

```python
draw_boundary(axes[2], wide_grid_scores, wide_title)
```

</details>

## 7. What changes with two hidden layers?

Our third experiment uses **5 neurons in the first hidden layer and 7 in the
second**, as in the lecture's diagram. We keep one scalar output because our
task is binary classification. Thus $L=3$ (including the output layer), with
$d_0=2$, $d_1=5$, $d_2=7$ and $d_3=1$.

The second hidden layer takes the first layer's activations as its inputs:

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,
&\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
\boldsymbol a_2(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2,
&\boldsymbol\phi_2(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_2(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_2(\boldsymbol x)\boldsymbol W_3+\boldsymbol b_3.
\end{aligned}
$$

Here $W_2,b_2$ belong to **hidden layer 2**; $W_3,b_3$ form the output layer.
The output is still an affine score, with no ReLU or sigmoid applied to it.
Both hidden layers' parameters will be fitted: the second layer combines learned
features to produce another learned representation of the original input.

In Python, prefix this experiment's names with `deep_`, for example `deep_W1` and
`deep_b1`. This keeps the earlier `net`, parameters and saved results available.
The prefix only identifies the experiment; the numbers still identify layers.
Use `deep_net(X)` for the new network. The loss function, mistake-count function,
dataset and plotting helper can all be reused without changes.

### 7.1. Construct the new architecture

**Exercise 12 — initialise all six parameter tensors.**

1. Use the matrix products above to derive the shapes of `deep_W1`, `deep_b1`,
   `deep_W2`, `deep_b2`, `deep_W3` and `deep_b3`. Keep biases one-dimensional, as
   before, so they are added to every observation's row.
2. Use `t.randn(..., requires_grad=True)` and the supplied dimension variables.
   Draw in the order **`deep_b1`, `deep_W1`, `deep_b2`, `deep_W2`, `deep_b3`,
   `deep_W3`** to reproduce the reference experiment.
3. Print all six shapes and calculate the total number of scalar parameters,
   using their dimensions or `.numel()`.

Keep the supplied RNG restoration and saved copies. As with the wider network,
the same bookmark and draw order make reruns reproducible; the different tensor
shapes mean that the three architectures have different initial parameter arrays.
Leave the previous exercises unchanged and run this new experiment in these cells.

In [ ]:
t.set_rng_state(parameter_initial_state)  # Supplied: reuse the bookmark after the data.
deep_input_features = 2
deep_hidden1_features = 5
deep_hidden2_features = 7
deep_output_features = 1

# YOUR CODE: create the six tensors in the stated order; print shapes and count entries.

# Supplied: keep independent starting values for all three layers.
deep_initial_b1 = deep_b1.detach().clone()
deep_initial_W1 = deep_W1.detach().clone()
deep_initial_b2 = deep_b2.detach().clone()
deep_initial_W2 = deep_W2.detach().clone()
deep_initial_b3 = deep_b3.detach().clone()
deep_initial_W3 = deep_W3.detach().clone()

<details>
<summary>Show answer — parameter tensors for two hidden layers</summary>

The shapes are `deep_W1`: `(2, 5)`, `deep_b1`: `(5,)`, `deep_W2`: `(5, 7)`,
`deep_b2`: `(7,)`, `deep_W3`: `(7, 1)` and `deep_b3`: `(1,)`.
There are $(2\times5+5)+(5\times7+7)+(7\times1+1)=65$ adjustable scalars.
The second matrix maps five hidden activations to seven preactivations; the last
matrix maps seven activations to one output score.

```python
t.set_rng_state(parameter_initial_state)  # Supplied: reuse the bookmark after the data.
deep_input_features = 2
deep_hidden1_features = 5
deep_hidden2_features = 7
deep_output_features = 1

deep_b1 = t.randn(deep_hidden1_features, requires_grad=True)
deep_W1 = t.randn(deep_input_features, deep_hidden1_features, requires_grad=True)
deep_b2 = t.randn(deep_hidden2_features, requires_grad=True)
deep_W2 = t.randn(deep_hidden1_features, deep_hidden2_features, requires_grad=True)
deep_b3 = t.randn(deep_output_features, requires_grad=True)
deep_W3 = t.randn(deep_hidden2_features, deep_output_features, requires_grad=True)

print('deep_W1:', deep_W1.shape)
print('deep_b1:', deep_b1.shape)
print('deep_W2:', deep_W2.shape)
print('deep_b2:', deep_b2.shape)
print('deep_W3:', deep_W3.shape)
print('deep_b3:', deep_b3.shape)
deep_parameter_count = (deep_W1.numel() + deep_b1.numel()
                        + deep_W2.numel() + deep_b2.numel()
                        + deep_W3.numel() + deep_b3.numel())
print('Scalar parameters:', deep_parameter_count)

# Supplied: keep independent starting values for all three layers.
deep_initial_b1 = deep_b1.detach().clone()
deep_initial_W1 = deep_W1.detach().clone()
deep_initial_b2 = deep_b2.detach().clone()
deep_initial_W2 = deep_W2.detach().clone()
deep_initial_b3 = deep_b3.detach().clone()
deep_initial_W3 = deep_W3.detach().clone()
```

</details>

**Exercise 13 — implement `deep_net(X)`.**

Translate the five equations into a function using the six `deep_` parameter
tensors. Use intermediate variables `a1`, `phi1`, `a2`, `phi2` and `scores`.
The input to the second hidden layer must be `phi1`, not the original `X`.
Return `scores`; do not draw parameters, detach tensors or convert them to
Python numbers inside the function.

Before running it, predict the shapes of all five intermediate tensors for the
full dataset and for the single-row slice `X[:1]`. How many feature values
describe each observation after the first hidden layer? After the second?

Use the existing `loss_from_scores` and `mistakes_from_scores` to report the
initial empirical loss and mistakes. Those measurements can be inside `no_grad`.
Explain why neither helper needs to know how many hidden layers there are.

In [ ]:
# YOUR CODE: define deep_net(X), then evaluate its initial loss and mistakes.

<details>
<summary>Show answer — composing two hidden layers</summary>

For the full dataset, `a1` and `phi1` have shape `(100, 5)`, `a2` and `phi2`
have shape `(100, 7)`, and `scores` has shape `(100, 1)`. With one input row,
replace the first dimension by 1. Each observation has five first-layer feature
values and seven second-layer feature values. All observations share the same
weights and biases.

The two helpers only need the scalar score for each observation and the labels
in `y`. They evaluate the same empirical loss and classification rule regardless
of how the network produced those scores. With the stated initialisation, the
initial loss is about $82.858$, with 29 mistakes.

```python
def deep_net(X):
    a1 = X @ deep_W1 + deep_b1
    phi1 = t.relu(a1)
    a2 = phi1 @ deep_W2 + deep_b2
    phi2 = t.relu(a2)
    scores = phi2 @ deep_W3 + deep_b3
    return scores

with t.no_grad():
    deep_initial_scores = deep_net(X)
    print('Initial empirical loss:', loss_from_scores(deep_initial_scores).item())
    print('Initial mistakes:', mistakes_from_scores(deep_initial_scores))
```

</details>

In [ ]:
# Supplied checks: the output remains one scalar score per observation.
assert deep_net(X).shape == y.shape
assert deep_net(X[:1]).shape == (1, 1)

### 7.2. Fit every layer

The supplied `restart_deep_network()` below restores the six saved tensors and
clears any stored gradients, just as `restart_network()` did for four tensors.
Run this definition before Exercise 14. We reuse the existing plotting grid;
it remains separate from the observations used in the empirical loss.

In [ ]:
# Supplied: reset this experiment without changing the earlier network.
def restart_deep_network():
    with t.no_grad():
        deep_b1.copy_(deep_initial_b1)
        deep_W1.copy_(deep_initial_W1)
        deep_b2.copy_(deep_initial_b2)
        deep_W2.copy_(deep_initial_W2)
        deep_b3.copy_(deep_initial_b3)
        deep_W3.copy_(deep_initial_W3)
    for parameter in (deep_b1, deep_W1, deep_b2, deep_W2, deep_b3, deep_W3):
        if parameter.grad is not None:
            parameter.grad.zero_()

**Exercise 14 — adapt your gradient-descent loop to six tensors.**

Use your own solution to Exercise 5 as a starting point for the scaffold below.
Keep the same 1,000 full-dataset updates and learning rate `0.001`.

1. Initialise `deep_loss_history` and `deep_error_history` with the measurements
   at update 0. Use ordinary Python numbers, as in the earlier histories.
2. Compute a fresh loss from `deep_net(X)` and call `.backward()` once per step.
   PyTorch differentiates the entire calculation, including both hidden layers.
3. Inside `no_grad`, update **all six** tensors using their gradients, then
   evaluate `deep_net(X)` again and append the new loss and mistake count.
4. Clear **all six** stored gradients before the next iteration.

Keep the supplied reset and grid snapshots. Use the `deep_` history names so
the earlier results remain available. Explain which parts of the earlier loop
change and which parts implement exactly the same mathematical operations.
In particular, what would go wrong if you forgot to update the output layer,
or if you computed the loss for `.backward()` inside `no_grad`?

After fitting, the supplied checks print progress at selected updates. Rerunning
Exercise 14 restores this model's saved starting values. To repeat its *initial*
evaluation in Exercise 13 after fitting, rerun Exercise 12 first.

In [ ]:
restart_deep_network()  # Supplied: restore all six starting tensors and clear gradients.
deep_learning_rate = 0.001
deep_num_updates = 1000
deep_boundary_steps = (1, 100, deep_num_updates)
deep_boundary_scores = {}

# YOUR CODE: initialise deep_loss_history and deep_error_history, including update 0.

for step in range(1, deep_num_updates + 1):
    # YOUR CODE: evaluate a fresh loss and compute its gradients.

    with t.no_grad():
        # YOUR CODE: update all six deep_ parameter tensors.
        # YOUR CODE: recompute scores and append loss and mistakes to the deep_ histories.

        # Supplied: store scores after this update, for later boundary plots.
        if step in deep_boundary_steps:
            deep_boundary_scores[step] = deep_net(X_grid).reshape(x1_grid.shape).clone()

    # YOUR CODE: clear all six stored gradients before the next iteration.

<details>
<summary>Show answer — fitting both hidden layers and the output</summary>

The model function, parameter names and number of tensors change. The empirical
loss, one differentiation call per step and gradient-descent rule stay the same.
All six derivatives are evaluated before any parameter is updated.

Forgetting the output update would keep that layer fixed, fitting only part of
the network. Computing the loss inside `no_grad` would prevent PyTorch from
tracking the calculation needed to obtain these derivatives.

```python
restart_deep_network()  # Supplied: restore all six starting tensors and clear gradients.
deep_learning_rate = 0.001
deep_num_updates = 1000
deep_boundary_steps = (1, 100, deep_num_updates)
deep_boundary_scores = {}

with t.no_grad():
    scores = deep_net(X)
    deep_loss_history = [loss_from_scores(scores).item()]
    deep_error_history = [mistakes_from_scores(scores)]

for step in range(1, deep_num_updates + 1):
    L = loss_from_scores(deep_net(X))
    L.backward()

    with t.no_grad():
        deep_b1.add_(-deep_learning_rate * deep_b1.grad)
        deep_W1.add_(-deep_learning_rate * deep_W1.grad)
        deep_b2.add_(-deep_learning_rate * deep_b2.grad)
        deep_W2.add_(-deep_learning_rate * deep_W2.grad)
        deep_b3.add_(-deep_learning_rate * deep_b3.grad)
        deep_W3.add_(-deep_learning_rate * deep_W3.grad)

        scores = deep_net(X)
        deep_loss_history.append(loss_from_scores(scores).item())
        deep_error_history.append(mistakes_from_scores(scores))

        # Supplied: store scores after this update, for later boundary plots.
        if step in deep_boundary_steps:
            deep_boundary_scores[step] = deep_net(X_grid).reshape(x1_grid.shape).clone()

    deep_b1.grad.zero_()
    deep_W1.grad.zero_()
    deep_b2.grad.zero_()
    deep_W2.grad.zero_()
    deep_b3.grad.zero_()
    deep_W3.grad.zero_()
```

</details>

In [ ]:
# Supplied checks and progress summary for the third experiment.
assert len(deep_loss_history) == len(deep_error_history) == deep_num_updates + 1
assert set(deep_boundary_scores) == set(deep_boundary_steps)
with t.no_grad():
    assert abs(loss_from_scores(deep_net(X)).item() - deep_loss_history[-1]) < 1e-4
    assert mistakes_from_scores(deep_net(X)) == deep_error_history[-1]
    assert t.allclose(deep_net(X_grid).reshape(x1_grid.shape),
                      deep_boundary_scores[deep_num_updates])
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, deep_num_updates):
    print(f'{step:7d}   {deep_loss_history[step]:23.6f}   {deep_error_history[step]:8d}')

### 7.3. Compare width and depth

**Exercise 15 — compare all three neural networks.**

The three short parts below reuse the plotting patterns you have already used.
All layouts, labels and reference curves are supplied. Complete only the marked
calls; `draw_boundary` remains supplied throughout.

**Part A: loss and mistakes.** Add the deeper network to the two panels: use
`deep_loss_history` on `axes[0]` and `deep_error_history` on `axes[1]`, both
against `updates`. Use `label='5 + 7 hidden neurons'` and solid lines (omit the
line-style argument). Keep the supplied colours: purple for loss, green for
mistakes. Each history includes updates 0 through 1,000.

In [ ]:
# Supplied: layout and the two earlier networks.
updates = range(deep_num_updates + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(updates, narrow_loss_history, ':', color='tab:purple', label='2 hidden neurons')
axes[0].plot(updates, wide_loss_history, '--', color='tab:purple', label='100 hidden neurons')
axes[1].plot(updates, narrow_error_history, ':', color='tab:green', label='2 hidden neurons')
axes[1].plot(updates, wide_error_history, '--', color='tab:green', label='100 hidden neurons')

# YOUR CODE START
# Add the deeper network's loss on axes[0] and mistakes on axes[1].
# YOUR CODE END

# Supplied: formatting and display.
axes[0].set_title('Empirical logistic loss')
axes[0].set_ylabel('Summed loss')
axes[1].set_title('Classification mistakes')
axes[1].set_ylabel('Mistakes')
for ax in axes:
    ax.set_xlabel('Completed updates')
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()

<details>
<summary>Show answer — adding the deeper network</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

Only the two selected histories and labels change; the plotting operations are
the same as in Exercise 11.

```python
axes[0].plot(updates, deep_loss_history, color='tab:purple', label='5 + 7 hidden neurons')
axes[1].plot(updates, deep_error_history, color='tab:green', label='5 + 7 hidden neurons')
```

</details>

**Part B: final boundaries.** Add one `draw_boundary` call on `axes[2]` for the
deeper network after all updates. Select its final grid scores from
`deep_boundary_scores`, using key `deep_num_updates`. Use the supplied
`deep_title`; the two earlier models are already drawn.

In [ ]:
# Supplied: layout, earlier boundaries and title.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
draw_boundary(axes[0], narrow_grid_scores, f'2 hidden neurons\nMistakes: {narrow_error_history[-1]}')
draw_boundary(axes[1], wide_grid_scores, f'100 hidden neurons\nMistakes: {wide_error_history[-1]}')
deep_title = f'5 + 7 hidden neurons\nMistakes: {deep_error_history[-1]}'

# YOUR CODE START
# Draw the deeper model's final boundary on axes[2], using deep_title.
# YOUR CODE END

# Supplied: display.
axes[0].legend(loc='upper left', fontsize=8)
plt.show()

<details>
<summary>Show answer — the deeper network after fitting</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

The dictionary key selects the completed update, not the panel number.

```python
draw_boundary(axes[2], deep_boundary_scores[deep_num_updates], deep_title)
```

</details>

**Part C: progress during fitting.** The supplied first panel shows the deeper
network after one update. Add two calls: after 100 updates on `axes[1]`, using
`deep_title_100`, and after 1,000 updates on `axes[2]`, using `deep_title_1000`.
Use `deep_boundary_scores[step]`, as in Exercise 7.

In [ ]:
# Supplied: layout, one-update example and titles.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
draw_boundary(axes[0], deep_boundary_scores[1], f'Deeper NN: 1 update\nMistakes: {deep_error_history[1]}')
deep_title_100 = f'Deeper NN: 100 updates\nMistakes: {deep_error_history[100]}'
deep_title_1000 = f'Deeper NN: 1000 updates\nMistakes: {deep_error_history[1000]}'

# YOUR CODE START
# Draw the saved scores after 100 updates on axes[1] and 1000 on axes[2].
# YOUR CODE END

# Supplied: display.
axes[0].legend(loc='upper left', fontsize=8)
plt.show()

<details>
<summary>Show answer — the two later stages</summary>

**Insert only these lines between `# YOUR CODE START` and `# YOUR CODE END`. Keep the supplied code above and below, then rerun the whole cell.**

These saved scores show how the boundary changed during fitting. Re-evaluating
the current network would instead show its final boundary each time.

```python
draw_boundary(axes[1], deep_boundary_scores[100], deep_title_100)
draw_boundary(axes[2], deep_boundary_scores[1000], deep_title_1000)
```

</details>

### Interpret the three experiments

The supplied table below collects the final measurements. The parameter counts
are those you derived in Exercises 1, 10 and 12. Use it together with your plots
to answer these questions before opening the interpretation:

1. How much does the deeper network improve between its first and its 1,000th
   update? Compare this with the change from update 0 to update 1.
2. How does it compare with the two single-hidden-layer networks, including the
   number of parameters and the shapes of the boundaries?
3. Have we isolated the effect of depth, given that widths, parameter counts and
   initial parameter arrays also differ?
4. Does a smaller empirical loss establish a global minimum or better predictions
   on unseen data? Does a flat mistake curve mean fitting has stopped?

In [ ]:
# Supplied: collect your final measurements for interpretation.
print('Hidden neurons       Parameters   Final loss   Mistakes')
print(f'2                             9   {narrow_loss_history[-1]:10.3f}   {narrow_error_history[-1]}')
print(f'100                         401   {wide_loss_history[-1]:10.3f}   {wide_error_history[-1]}')
print(f'5 then 7                     65   {deep_loss_history[-1]:10.3f}   {deep_error_history[-1]}')

<details>
<summary>Show answer — interpreting width and depth</summary>

The deeper network makes 29 mistakes initially and after one update, 13 after
100 updates and 2 after 1,000. Its empirical loss falls from about $82.858$ to
$76.103$ in the first update, and to $6.493$ by the end.

Its 65 parameters give a better fit in this run than the 9-parameter network
(loss $25.196$, 10 mistakes). The 401-parameter network fits best here
(loss $4.713$, 1 mistake). Both larger networks' boundaries follow the clouds
more closely than the two-neuron fit.

The comparison does not isolate depth: widths, parameter counts and initial
values also differ. None of these results proves a global minimum or better
prediction on unseen observations. A mistake count can stay constant while the
scores and empirical loss change, because a label changes only when its score
crosses the zero threshold.

</details>

You have translated a network architecture into tensors, implemented its score
function, calculated empirical losses and mistakes, obtained derivatives, and
used them in your own gradient-descent loop. Your curves and boundary snapshots
let you compare the fitted network with the affine model on the same data. You
also inspected how changing hidden parameters changes the feature representation,
compared ideal XOR inputs with noisy observations, widened the network, and
implemented a second hidden layer. The three experiments kept the dataset,
empirical loss, learning rate and update count fixed, while changing how the
network constructs its learned features.

This completes the neural-network fitting experiment.